# 🏆 Notebook 4 — Your Turn: Explore & Build

You've built an agent (Notebook 1), added skills (Notebook 2), and deployed to AgentCore (Notebook 3).

Now it's your turn. This notebook has starting points — pick what interests you and go deep.

---
## Setup (run this first)

In [ ]:
!pip install --quiet strands-agents mcp boto3 python-dotenv bedrock-agentcore 2>/dev/null
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['PATH'] = f"{os.path.expanduser('~')}/.local/bin:{os.environ['PATH']}"

from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

model = BedrockModel(model_id='us.anthropic.claude-sonnet-4-5-20250929-v1:0', streaming=False)
server_params = StdioServerParameters(
    command='uvx',
    args=['teradata-mcp-server'],
    env={
        'DATABASE_URI': os.getenv('TERADATA_DATABASE_URI'),
        'LOGMECH': os.getenv('LOGMECH', 'TD2')
    }
)
teradata_tool = MCPClient(lambda: stdio_client(server_params), startup_timeout=300)
print('Ready.')

---
## Challenge 1: Build Your Own Specialized Agent

Same MCP connection, same model — completely different agent. Just change the system prompt.

**Ideas:**
- **Claims Analyst** — find billing anomalies in WORKERS_COMP_HDR/DTL
- **Compliance Officer** — flag PAs missing required documentation
- **Member Advocate** — look up a member's full history across PAs and claims
- **Provider Profiler** — compare provider patterns, specialties, alert history

Edit the prompt below and try it:

In [ ]:
# Change this prompt to create your own agent
my_prompt = """
You are a [YOUR ROLE] for a healthcare organization.
You have access to the HCLS Teradata database.

Your focus:
- [WHAT YOU CARE ABOUT]
- [WHAT METRICS MATTER]
- [WHAT ACTIONS TO RECOMMEND]

Use Teradata SQL (TOP instead of LIMIT). Show data to support your analysis.
"""

my_agent = Agent(model=model, tools=[teradata_tool], system_prompt=my_prompt)

# Ask it something
# result = my_agent('Your question here')
# print(result.message)

---
## Challenge 2: Add a New Skill

In Notebook 2 you saw skills for dashboard, care pathway, and fraud check.
Create a new one. Some ideas:

- **Denial Predictor** — check if a PA would likely be denied based on missing prerequisites
- **Cost Estimator** — look up claims history for the requested CPT codes and estimate cost
- **Peer Comparison** — compare a provider's PA patterns to others in the same specialty

Template:

In [ ]:
my_skill = """
## [Your Skill Name]
When the user asks about [trigger topic]:

1. Query [table] to get [data]
2. Query [table] to check [condition]
3. Present results with [format]
"""

# Use it
# agent = Agent(model=model, tools=[teradata_tool], system_prompt=BASE_PROMPT + my_skill)
# result = agent('Your question')
# print(result.message)

---
## Challenge 3: Explore AgentCore Features

You deployed a basic agent in Notebook 3. AgentCore has much more:

### Gateway — Governed tool access
Add auth, policy, and audit to your MCP connections.  
📖 [AgentCore Gateway docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)

### Memory — Persistent context
Give your agent memory across sessions (semantic, summarization, user preferences).  
📖 [AgentCore Memory docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)

### Identity — User authentication
Connect to identity providers so the agent knows who's asking.  
📖 [AgentCore Identity docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity.html)

### Evaluations — Is the agent good?
Score agent responses automatically with LLM-as-a-judge.  
📖 [AgentCore Evaluations docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html)

### Observability — Traces and logs
Debug agent behavior with full traces.  
📖 [AgentCore Observability docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)

### Code Interpreter — Run code
Let the agent write and execute Python for analysis.  
📖 [AgentCore Code Interpreter docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/code-interpreter.html)

---
## What You Built Today

```
Raw Bedrock call        →  "I can call a model"
  + MCP (Teradata)      →  "It can query real data"
    + Strands Agent     →  "It reasons and iterates"
      + Skills          →  "It handles different tasks"
        + AgentCore     →  "It runs in production"
```

### Resources

- [AgentCore Developer Guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/)
- [AgentCore CLI (GitHub)](https://github.com/aws/agentcore-cli)
- [AgentCore Samples (GitHub)](https://github.com/awslabs/agentcore-samples)
- [Strands Agents SDK](https://strandsagents.com)
- [Teradata MCP Server](https://github.com/Teradata/teradata-mcp-server)

---
## Clean Up

When you're done, run `teardown.sh` from your local machine to delete the SageMaker notebook.

In [ ]:
# Delete AgentCore deployment (uncomment to run)
# !cd ~/SageMaker/claire-agentcore && agentcore destroy